[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/labs/L4/lab.ipynb)

# Lab 4 · SVM y comparativa final
**DAMMG · Módulo VI · Sesión 4** · unos 45 min de trabajo · entrega en Moodle: **dom 15 nov, 23:59** (tolerancia hasta el mié 18 nov)

**Al terminar podrás**
- **(O5.1)** predecir cómo cambian el margen y los vectores de soporte al mover $\textcolor{#f27900}{C}$ (y la frontera al mover $\textcolor{#f27900}{\gamma}$), y explicarlo con la función hinge;
- **(O5.2)** verificar numéricamente que $\phi(x)\cdot\phi(z) = (x\cdot z+1)^2$;
- **(O6.1)** comparar todos los modelos del módulo en MiBici con la misma métrica y decidir si una diferencia es real o ruido.

**Ruta (PRIMM).** L0 preparar, L1 ejemplo resuelto (margen duro), L2 investigar $C$ y $\gamma$ con sliders, L3 completar funciones, L4 cazar errores, L5 MiBici: SVM en muestra, `LinearSVC` y `SGDClassifier` en todo train, tabla final, L6 mini-reto.

> **Carril exprés (L3, L5 y L6).** Si ya dominas los SVM, completa L3 (es lo que se califica) y salta a L5–L6. Ejecuta igual **todas** las celdas en orden aunque no las leas: varias autocomprobaciones usan variables de celdas anteriores.

**Qué se califica.** Las celdas `graded`: `hinge`, `kernel_poli2` y `margen` (L3). Reglas: nombres y argumentos intactos; dentro de las funciones solo `numpy` y `math`, sin variables globales, sin `du.` y sin funciones de otras celdas (el evaluador solo ve las celdas `graded`; la autocomprobación te avisa si algo se escapa). **Cuánto vale:** cada lab es 7.5 % del módulo; tu nota es la fracción de pruebas del evaluador que pasan tus funciones.

**Fuentes.** Andrew Ng, *Machine Learning* (Coursera, 2011), semana 7 (SVM, kernels). Stanford CS229 (2018), clases 6 y 7 y *lecture notes* 3. Patrick Winston, MIT 6.034, clase 16 (CC BY-NC-SA). Abu-Mostafa, Caltech CS156, clases 14 y 15.

## L0 · Preparación

> **Antes de escribir nada**
> 1. *Archivo*, opción *Guardar una copia en Drive* y trabaja en esa copia; si no, al cerrar la pestaña pierdes todo.
> 2. Ejecuta las celdas **en orden**. Si reinicias el entorno o algo falla raro: *Entorno de ejecución*, opción *Reiniciar y ejecutar todo*.
> 3. Escribe dentro de las celdas que ya existen (no copies/pegues celdas): así no pierden la etiqueta que usa el evaluador.
>
> *Palabras en inglés que usamos tal cual:* **baseline** (línea base: el modelo tonto que hay que vencer) · **pipeline** (cadena de preprocesamiento + modelo) · **proxy** (variable sustituta de lo que de verdad queremos medir) · **bootstrap** (remuestreo con reemplazo) · **chat waterfall** (todos escriben su respuesta en el chat y la envían al mismo tiempo) · **precision** y **recall** (los nombres de sklearn para precisión y exhaustividad).

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dammg_utils import COLORES
from sklearn.svm import SVC, LinearSVC
du.estilo_dammg()
np.set_printoptions(precision=4, suppress=True)

> ### ✅ Reacción en Zoom · fin de L0
> Reacciona con ✅ cuando veas `✅ entorno listo`.
> Si te atoraste, reacciona con ✋ y escribe en el chat el número de la celda.

## L1 · Ejemplo resuelto: la calle más ancha
Dos grupos que se pueden separar con una recta. De todas las rectas que separan, el SVM elige la que deja la **calle** más ancha (Winston). Con etiquetas $y\in\{-1,+1\}$, el margen duro resuelve
$$\min_{\textcolor{#0847c4}{\vec w},b}\ \tfrac12\lVert \textcolor{#0847c4}{\vec w}\rVert^2 \quad\text{sujeto a}\quad y^{(i)}(\textcolor{#0847c4}{\vec w}\cdot x^{(i)}+b)\ge 1.$$
Las orillas de la calle son los planos $\textcolor{#0847c4}{\vec w}\cdot x+b=\pm1$ y su distancia es $2/\lVert \textcolor{#0847c4}{\vec w}\rVert$. Los puntos que tocan las orillas son los **vectores de soporte**. Con `SVC(kernel="linear")` y un $C$ enorme obtenemos el margen duro.

In [ ]:
rng = np.random.default_rng(6)
X_sep = np.vstack([rng.normal([2.0, 2.0], 0.6, (15, 2)), rng.normal([5.0, 4.5], 0.6, (15, 2))])
y_sep = np.r_[-np.ones(15), np.ones(15)]


def dibuja_svm(ax, modelo, X, y, titulo="", lim=None):
    lim = lim or (X[:, 0].min() - 1, X[:, 0].max() + 1, X[:, 1].min() - 1, X[:, 1].max() + 1)
    xx, yy = np.meshgrid(np.linspace(lim[0], lim[1], 200), np.linspace(lim[2], lim[3], 200))
    f = modelo.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, f, levels=[-1e9, 0, 1e9], colors=[COLORES["datos"], "#fde3e4"], alpha=0.6)
    ax.contour(xx, yy, f, levels=[-1, 0, 1], colors=[COLORES["tinta"], COLORES["w"], COLORES["tinta"]],
               linestyles=["--", "-", "--"], linewidths=[1.5, 2.5, 1.5])
    ax.scatter(*X[y < 0].T, s=45, marker="o", color=COLORES["datos"], edgecolors=COLORES["tinta"], label="y = −1")
    ax.scatter(*X[y > 0].T, s=55, marker="X", color=COLORES["error"], label="y = +1")
    sv = modelo.support_vectors_
    ax.scatter(*sv.T, s=220, facecolors="none", edgecolors=COLORES["amarillo"], linewidths=2.5, label="vector de soporte")
    ax.set(xlim=lim[:2], ylim=lim[2:], title=titulo)


duro = SVC(kernel="linear", C=1e4).fit(X_sep, y_sep)
w_d, b_d = duro.coef_[0], duro.intercept_[0]
print(f"w = {w_d.round(3)}, b = {b_d:.3f}, ‖w‖ = {np.linalg.norm(w_d):.3f}, ancho de la calle 2/‖w‖ = {2 / np.linalg.norm(w_d):.3f}")
print("y·f(x) de los vectores de soporte:", (y_sep[duro.support_] * duro.decision_function(X_sep[duro.support_])).round(3))
print(f"mínimo de y·f(x) en todos los puntos: {np.min(y_sep * duro.decision_function(X_sep)):.3f}")

fig, ax = plt.subplots(figsize=(6, 4.8))
dibuja_svm(ax, duro, X_sep, y_sep, f"Margen duro · {len(duro.support_)} vectores de soporte")
ax.legend(loc="upper left", fontsize=8)
plt.show()

**Antes de ejecutar.** Vamos a borrar un punto y reentrenar. (a) Si borras un punto que **no** es vector de soporte, ¿se mueve la frontera? (b) ¿Y si borras un vector de soporte?

**Tu predicción:** _(escribe aquí)_

In [ ]:
no_sv = np.setdiff1d(np.arange(len(y_sep)), duro.support_)
for nombre, quitar in [("un punto que NO es vector de soporte", no_sv[0]), ("un vector de soporte", duro.support_[0])]:
    quedan = np.delete(np.arange(len(y_sep)), quitar)
    m = SVC(kernel="linear", C=1e4).fit(X_sep[quedan], y_sep[quedan])
    print(f"sin {nombre:38s}: w = {m.coef_[0].round(3)}, b = {m.intercept_[0]:7.3f}, ancho = {2 / np.linalg.norm(m.coef_[0]):.3f}")

La frontera depende **solo** de los vectores de soporte: los demás puntos podrían no existir. Al quitar un vector de soporte la calle se ensancha (hay menos restricciones). Esa es la "esparsidad" del SVM, y es lo que más adelante permite el truco del kernel: la predicción solo necesita productos punto con unos cuantos puntos.

> ### ✅ Reacción en Zoom · fin de L1
> Escribe en el chat el ancho de la calle (dos decimales).
> Si te atoraste, reacciona con ✋ y escribe en el chat el número de la celda.

## L2 · Investiga $C$ y $\gamma$
Datos reales casi nunca se separan. El **margen suave** permite violaciones y las cobra con la hinge:
$$\min_{\textcolor{#0847c4}{\vec w},b}\ \tfrac12\lVert \textcolor{#0847c4}{\vec w}\rVert^2 + \textcolor{#f27900}{C}\sum_i \max\!\big(0,\ 1-y^{(i)}(\textcolor{#0847c4}{\vec w}\cdot x^{(i)}+b)\big).$$
$\textcolor{#f27900}{C}$ grande: cada violación sale cara, calle angosta. $\textcolor{#f27900}{C}$ chico: calle ancha y muchos puntos dentro. Con el kernel RBF, $K(x,z)=e^{-\gamma\lVert x-z\rVert^2}$, el parámetro $\textcolor{#f27900}{\gamma}$ decide qué tan lejos llega la influencia de cada punto.

Usamos dos "lunas" con ruido: 150 puntos para entrenar y 150 para validar.

In [ ]:
from sklearn.datasets import make_moons

X_lunas, y_lunas01 = make_moons(n_samples=300, noise=0.3, random_state=0)
y_lunas = 2 * y_lunas01 - 1                        # de {0, 1} a {−1, +1}
X_l_tr, X_l_va, y_l_tr, y_l_va = X_lunas[:150], X_lunas[150:], y_lunas[:150], y_lunas[150:]
LIM_LUNAS = (-2, 3, -1.6, 2.1)

**Antes de mover los sliders.** Con kernel lineal, si $C$ sube de 0.01 a 100: ¿la calle se ensancha o se angosta? ¿Hay más o menos vectores de soporte? Con RBF, ¿qué esperas ver con $\gamma$ = 100?

**Tu predicción:** _(escribe aquí)_

In [ ]:
def explora_svm(kernel="rbf", log10_C=0.0, log10_gamma=0.0):
    C, gamma = 10.0 ** log10_C, 10.0 ** log10_gamma
    modelo = SVC(kernel="linear", C=C) if kernel == "lineal" else SVC(kernel="rbf", C=C, gamma=gamma)
    modelo.fit(X_l_tr, y_l_tr)
    acc_tr = np.mean(modelo.predict(X_l_tr) == y_l_tr)
    acc_va = np.mean(modelo.predict(X_l_va) == y_l_va)
    extra = f" · ancho {2 / np.linalg.norm(modelo.coef_[0]):.2f}" if kernel == "lineal" else f", γ = {gamma:g}"
    fig, ax = plt.subplots(figsize=(6.5, 4.8))
    dibuja_svm(ax, modelo, X_l_tr, y_l_tr, f"C = {C:g}{extra}\nacc train {acc_tr:.2f} · val {acc_va:.2f} · {len(modelo.support_)} VS",
               lim=LIM_LUNAS)
    plt.show()


try:
    import ipywidgets as widgets
    widgets.interact(explora_svm,
                     kernel=widgets.Dropdown(options=["rbf", "lineal"], value="rbf", description="kernel"),
                     log10_C=widgets.FloatSlider(value=0.0, min=-2, max=3, step=0.25, description="log10 C"),
                     log10_gamma=widgets.FloatSlider(value=0.0, min=-2, max=2, step=0.25, description="log10 γ"))
except ImportError:
    print("No hay ipywidgets: usa la celda de abajo.")

In [ ]:
# Alternativa sin widget: cambia y re-ejecuta (prueba gamma 0.1, 1, 10, 100 y C 0.01, 1, 100)
explora_svm(kernel="rbf", log10_C=0.0, log10_gamma=0.0)

**Ahora en números.** La tabla barre $C$ con kernel lineal y $\gamma$ con RBF ($C=1$).

In [ ]:
filas = []
for C in [0.01, 0.1, 1, 10, 100]:
    m = SVC(kernel="linear", C=C).fit(X_l_tr, y_l_tr)
    filas.append({"kernel": "lineal", "C": C, "γ": "no aplica", "ancho 2/‖w‖": 2 / np.linalg.norm(m.coef_[0]),
                  "VS": len(m.support_), "acc train": np.mean(m.predict(X_l_tr) == y_l_tr),
                  "acc val": np.mean(m.predict(X_l_va) == y_l_va)})
for g in [0.1, 1, 10, 100]:
    m = SVC(kernel="rbf", C=1, gamma=g).fit(X_l_tr, y_l_tr)
    filas.append({"kernel": "rbf", "C": 1, "γ": g, "ancho 2/‖w‖": np.nan, "VS": len(m.support_),
                  "acc train": np.mean(m.predict(X_l_tr) == y_l_tr), "acc val": np.mean(m.predict(X_l_va) == y_l_va)})
tabla_lunas = pd.DataFrame(filas)
tabla_lunas.round(3)

Con kernel lineal, subir $C$ angosta la calle y baja el número de vectores de soporte: la hinge de cada punto dentro de la calle pesa más, así que el optimizador prefiere una $\lVert w\rVert$ más grande con menos violaciones. Con RBF, $\gamma$ chico da una frontera casi recta (**sesgo**) y $\gamma$ grande dibuja **islas** alrededor de los puntos de entrenamiento: train sube y validación baja (**varianza**). $C$ y $\gamma$ se eligen juntos, en escala logarítmica, con validación.

> ### ✅ Reacción en Zoom · fin de L2
> Chat waterfall: ¿qué γ te dio islas? Escribe el valor y la accuracy de val.
> Si te atoraste, reacciona con ✋ y escribe en el chat el número de la celda.

## L3 · Completa las funciones (esto se califica)
$$\text{hinge}(y,f)=\max(0,\,1-yf)\qquad K(x,z)=(x\cdot z+1)^2\qquad \text{margen}=\frac{2}{\lVert \textcolor{#0847c4}{\vec w}\rVert}$$

In [ ]:
import numpy as np


def hinge(y, f):
    """Pérdida hinge elemento por elemento: max(0, 1 − y·f).

    y : etiquetas en {−1, +1} (número o arreglo).   f : salida del modelo f(x) = w·x + b (misma forma que y).
    Vale 0 si el punto está bien clasificado y fuera del margen (y·f ≥ 1); crece en línea recta si no.
    Regresa un arreglo float con la forma de y·f (o un float si las entradas son números).
    """
    y = np.asarray(y, dtype=float)
    f = np.asarray(f, dtype=float)

    # TODO: una línea con np.maximum. Si el resultado es un número (ndim == 0) regresa float(...).
    raise NotImplementedError("Completa hinge (L3): busca el # TODO.")

In [ ]:
du.revisa_aislada(hinge, graded=['kernel_poli2', 'margen'])
# Autocomprobación de hinge
h = hinge(np.array([1, 1, 1, -1, -1]), np.array([2.0, 1.0, 0.3, -0.5, 2.0]))
assert np.shape(h) == (5,), "Debe regresar un arreglo con la forma de las entradas."
assert np.allclose(h, [0.0, 0.0, 0.7, 0.5, 3.0]), f"Esperaba [0, 0, 0.7, 0.5, 3] y obtuve {h}"
assert isinstance(hinge(1, 0.25), float) and np.isclose(hinge(1, 0.25), 0.75), "Con números regresa un float."
print("✅ hinge pasa la autocomprobación")

In [ ]:
def kernel_poli2(x, z):
    """Kernel polinomial de grado 2: K(x, z) = (x·z + 1)².

    x, z : vectores 1D de la misma longitud n (listas o arreglos). Regresa un float.
    Es el producto punto φ(x)·φ(z) de un espacio con todos los monomios de grado ≤ 2,
    pero sin construir φ: cuesta O(n) en lugar de O(n²).
    """
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)

    # TODO: producto punto, más 1, al cuadrado.
    raise NotImplementedError("Completa kernel_poli2 (L3): busca el # TODO.")

In [ ]:
du.revisa_aislada(kernel_poli2, graded=['hinge', 'margen'])
# Autocomprobación de kernel_poli2
assert np.isclose(kernel_poli2([1, 2], [3, -1]), 4.0), "(1·3 + 2·(−1) + 1)² = 2² = 4"
assert np.isclose(kernel_poli2([0, 0, 0], [5, 5, 5]), 1.0), "Con x = 0 vale (0 + 1)² = 1"
assert isinstance(kernel_poli2([1, 2], [3, 4]), float), "Regresa un float."
print("✅ kernel_poli2 pasa la autocomprobación")

In [ ]:
def margen(w):
    """Ancho del margen de un SVM lineal: distancia entre los planos w·x + b = +1 y w·x + b = −1.

    w : vector de pesos (lista o arreglo, también acepta forma (1, n) como coef_ de sklearn).
    Regresa 2 / ‖w‖ como float (‖·‖ es la norma euclidiana). No depende de b.
    """
    w = np.asarray(w, dtype=float).ravel()

    # TODO: usa np.linalg.norm.
    raise NotImplementedError("Completa margen (L3): busca el # TODO.")

In [ ]:
du.revisa_aislada(margen, graded=['hinge', 'kernel_poli2'])
# Autocomprobación de margen
assert np.isclose(margen([3.0, 4.0]), 0.4), "‖(3, 4)‖ = 5, así que 2/5 = 0.4"
assert np.isclose(margen(duro.coef_), 2 / np.linalg.norm(duro.coef_)), "Acepta la forma (1, n) de coef_ (usa .ravel())."
# Geometría: proyecta la diferencia entre un vector de soporte de cada clase sobre la dirección de w
sv_pos = X_sep[duro.support_][y_sep[duro.support_] > 0][0]
sv_neg = X_sep[duro.support_][y_sep[duro.support_] < 0][0]
distancia = (sv_pos - sv_neg) @ w_d / np.linalg.norm(w_d)
assert np.isclose(margen(w_d), distancia, atol=1e-3), f"La calle mide {distancia:.4f} y tu margen da {margen(w_d):.4f}"
print(f"✅ margen = {margen(w_d):.4f} coincide con la distancia medida entre las orillas ({distancia:.4f})")

### Verifica el truco del kernel (O5.2)
Para $x=(x_1,x_2)$, el mapeo $\phi(x) = (x_1^2,\ x_2^2,\ \sqrt2 x_1x_2,\ \sqrt2 x_1,\ \sqrt2 x_2,\ 1)$ cumple $\phi(x)\cdot\phi(z) = (x\cdot z+1)^2$. Desarrolla el cuadrado en papel si quieres verlo término a término (es la demo 7). Aquí lo comprobamos con números.

In [ ]:
x_k, z_k = np.array([1.0, 2.0]), np.array([3.0, -1.0])
print(f"φ(x) = {du.phi_poli2(x_k).round(4)}")
print(f"φ(z) = {du.phi_poli2(z_k).round(4)}")
print(f"φ(x)·φ(z) = {du.phi_poli2(x_k) @ du.phi_poli2(z_k):.6f}   ·   (x·z + 1)² = {kernel_poli2(x_k, z_k):.6f}")
rng_k = np.random.default_rng(0)
pares = rng_k.normal(size=(1000, 2, 2))
dif = max(abs(du.phi_poli2(a) @ du.phi_poli2(b) - kernel_poli2(a, b)) for a, b in pares)
assert dif < 1e-9, f"Diferencia máxima {dif}: revisa kernel_poli2"
print(f"✅ 1000 pares al azar: diferencia máxima {dif:.1e}")
for n in [2, 14, 1000]:
    print(f"n = {n:5d} variables: φ tiene {(n + 1) * (n + 2) // 2:>7d} coordenadas; el kernel solo hace un producto punto de {n}")

> ### ✅ Reacción en Zoom · fin de L3
> Reacciona con ✅ cuando las cuatro autocomprobaciones digan ✅.
> Si te atoraste, reacciona con ✋ y escribe en el chat el número de la celda.

## L4 · Caza de errores: dos tropiezos clásicos con SVM en MiBici
Cargamos el caso de siempre: $y=1$ si $|$`future_net`$|\ge 6$ (umbral congelado con train) y las variables del Lab 3. Un compañero quiere vigilar la pérdida hinge de su SVM y escribió esto. **Tiene dos errores.** Encuéntralos.

In [ ]:
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_parquet(RUTA_DATOS)
tr = df[df["particion"] == "train"].copy()
va = df[df["particion"] == "val"].copy()
te = df[df["particion"] == "test"].copy()
UMBRAL = UMBRAL_EVENTO                            # celda L0: percentil 85 de |future_net| en train, congelado
assert UMBRAL == float(tr["future_net"].abs().quantile(0.85))
y_tr, y_va, y_te = ((d["future_net"].abs() >= UMBRAL).astype(int).to_numpy() for d in (tr, va, te))

HORAS_PICO = [7, 8, 9, 17, 18, 19, 20]


def variables(d):
    # Mismas variables del Lab 3: rezagos ≥ 24 h, calendario y capacidad (nada de la hora t)
    return pd.DataFrame({
        "net_lag_24h": d["net_lag_24h"], "net_lag_168h": d["net_lag_168h"], "net_mean_24h": d["net_mean_24h"],
        "abs_net_lag_24h": d["net_lag_24h"].abs(), "abs_net_lag_168h": d["net_lag_168h"].abs(),
        "abs_net_mean_24h": d["net_mean_24h"].abs(),
        "departures_mean_24h": d["departures_mean_24h"], "departures_mean_168h": d["departures_mean_168h"],
        "departures_lag_168h": d["departures_lag_168h"], "capacity": d["capacity"],
        "is_weekend": d["is_weekend"], "pico": d["hour_of_day"].isin(HORAS_PICO).astype(int),
        "hour_of_day": d["hour_of_day"], "day_of_week": d["day_of_week"],
    }, index=d.index)


X_tr, X_va, X_te = variables(tr), variables(va), variables(te)
# Muestra estratificada de train para el SVM con kernel · 8 000 filas con la misma prevalencia
i_muestra, _ = train_test_split(np.arange(len(y_tr)), train_size=8000, stratify=y_tr, random_state=0)
print(f"muestra: {len(i_muestra)} filas, {y_tr[i_muestra].sum()} positivos (prevalencia {y_tr[i_muestra].mean():.3f})")

In [ ]:
# ---- código del compañero (no lo copies) ----
svm_bug = SVC(kernel="rbf", C=1, gamma=0.05).fit(X_tr.iloc[i_muestra], y_tr[i_muestra])   # variables sin escalar
f_bug = svm_bug.decision_function(X_tr.iloc[i_muestra])
perdida = du.hinge(y_tr[i_muestra], f_bug)                                                # etiquetas 0/1
print(f"hinge promedio = {perdida.mean():.3f}")
print(f"hinge promedio de los negativos = {perdida[y_tr[i_muestra] == 0].mean():.3f}  (¿por qué exactamente 1?)")

# Pista para el segundo error: ¿qué variable domina la distancia ‖x − z‖² que usa el RBF?
dif = X_tr.iloc[i_muestra[:500]].to_numpy() - X_tr.iloc[i_muestra[500:1000]].to_numpy()
aporte = pd.Series((dif ** 2).mean(axis=0), index=X_tr.columns)
print("\nParte de ‖x − z‖² que aporta cada variable (sin escalar):")
print((aporte / aporte.sum()).sort_values(ascending=False).head(4).round(3).to_string())

**¿Cuáles son los dos errores?** Pista: uno está en las etiquetas y el otro en la escala de las variables.

**Tu respuesta:** _(escribe aquí)_

> ### ✅ Reacción en Zoom · fin de L4
> Escribe en el chat los dos errores en pocas palabras.
> Si te atoraste, reacciona con ✋ y escribe en el chat el número de la celda.

## L5 · MiBici: SVM y la comparativa de todo el módulo
### L5a · SVM con kernel RBF en la muestra
`SVC` cuesta entre $O(m^2)$ y $O(m^3)$: con 254 mil filas de train no termina en clase. Por eso usamos la muestra estratificada de 8 000 filas. El escalador se ajusta con **todo train** (ajustarlo con la muestra también sería válido; nunca con validación). Para ahorrar tiempo, la rejilla de $C$ y $\gamma$ se compara en una submuestra estratificada de 10 000 filas de validación; el ganador se evalúa después en validación completa. Medimos el tiempo de cada ajuste: debe quedar por debajo de 60 s en la CPU de Colab. La rejilla de 6 SVC tarda de 1 a 3 min en Colab gratuito.

In [ ]:
import time

escalador = StandardScaler().fit(X_tr)                      # solo train
Z_tr, Z_va, Z_te = (escalador.transform(X) for X in (X_tr, X_va, X_te))
i_va_rejilla, _ = train_test_split(np.arange(len(y_va)), train_size=10_000, stratify=y_va, random_state=0)

rejilla = []
for C in [0.1, 1, 10]:
    for gamma in [0.01, 0.1]:
        t0 = time.perf_counter()
        m = SVC(kernel="rbf", C=C, gamma=gamma).fit(Z_tr[i_muestra], y_tr[i_muestra])
        segundos = time.perf_counter() - t0
        ap = average_precision_score(y_va[i_va_rejilla], m.decision_function(Z_va[i_va_rejilla]))
        rejilla.append({"C": C, "γ": gamma, "segundos de ajuste": segundos, "VS": int(m.n_support_.sum()), "AP val (submuestra)": ap})
tabla_rejilla = pd.DataFrame(rejilla)
if tabla_rejilla["segundos de ajuste"].max() > 60:
    print("⚠ Algún ajuste pasó de 60 s: si tu Colab va lento, baja la muestra a 4 000 filas.")
tabla_rejilla.round(3)

In [ ]:
mejor = tabla_rejilla.loc[tabla_rejilla["AP val (submuestra)"].idxmax()]
svc_rbf = SVC(kernel="rbf", C=mejor["C"], gamma=mejor["γ"]).fit(Z_tr[i_muestra], y_tr[i_muestra])
s_va_svc = svc_rbf.decision_function(Z_va)
print(f"SVC RBF elegido: C = {mejor['C']:g}, γ = {mejor['γ']:g} · AP val completa = {average_precision_score(y_va, s_va_svc):.3f}")

### L5b · SVM lineal con todo train
⚠ **Convención.** Los tres estimadores de SVM de scikit-learn no minimizan exactamente lo mismo:

| Estimador | Pérdida | ¿Regulariza $b$? | Nota |
|---|---|---|---|
| `SVC` | hinge | no | kernel; $C$ = peso del error, igual que en la fórmula de L2 |
| `LinearSVC` | **`squared_hinge` por defecto**; aquí fijamos `loss="hinge"` | **sí**: trata $b$ como un peso más sobre una columna constante `intercept_scaling` | escala a muchos ejemplos |
| `SGDClassifier(loss="hinge")` | hinge | no (el intercepto no se penaliza) | su `alpha` corresponde a $\approx 1/(m\,C)$ |

Regularizar $b$ casi no importa con variables estandarizadas (centradas en 0), pero puede arruinar el modelo si los datos están lejos del origen. Míralo con los puntos de L1 desplazados 20 unidades:

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

X_lejos = X_sep + 20
ref = SVC(kernel="linear", C=1).fit(X_lejos, y_sep)
print(f"SVC                                   b = {ref.intercept_[0]:8.3f} · acc = {np.mean(ref.predict(X_lejos) == y_sep):.2f}")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    for esc_b in [1, 100]:
        lin = LinearSVC(C=1, loss="hinge", dual=True, intercept_scaling=esc_b, max_iter=100_000, random_state=0).fit(X_lejos, y_sep)
        print(f"LinearSVC(loss='hinge', intercept_scaling={esc_b:<3}) b = {lin.intercept_[0]:8.3f} · acc = {np.mean(lin.predict(X_lejos) == y_sep):.2f}")

Con `intercept_scaling=1` la penalización empuja $b$ hacia 0 y la recta ya no puede llegar hasta los datos. Al subirlo a 100, el $b$ efectivo casi no se penaliza y recuperamos la solución de `SVC`. En MiBici las variables están estandarizadas, así que usamos el valor por defecto.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import SGDClassifier

C_lin = 0.1
t0 = time.perf_counter()
with warnings.catch_warnings():
    # LinearSVC a veces avisa que no convergió del todo: aquí no cambia la AP en el tercer decimal
    warnings.simplefilter("ignore", ConvergenceWarning)
    lin_svc = LinearSVC(C=C_lin, loss="hinge", dual=True, max_iter=20_000, random_state=0).fit(Z_tr, y_tr)
t_lin = time.perf_counter() - t0
t0 = time.perf_counter()
sgd = SGDClassifier(loss="hinge", alpha=1 / (len(y_tr) * C_lin), average=True, max_iter=30, tol=None, random_state=0).fit(Z_tr, y_tr)
t_sgd = time.perf_counter() - t0
s_va_lin, s_va_sgd = lin_svc.decision_function(Z_va), sgd.decision_function(Z_va)
print(f"LinearSVC (hinge, C = {C_lin}) en {len(y_tr)} filas: {t_lin:.1f} s · AP val = {average_precision_score(y_va, s_va_lin):.3f}")
print(f"SGDClassifier (hinge, alpha = 1/(m·C)): {t_sgd:.1f} s · AP val = {average_precision_score(y_va, s_va_sgd):.3f}")

### L5c · La tabla final: todos los modelos del módulo
Mismos datos, misma partición, misma métrica (AP en validación). Los hiperparámetros de árbol, bosque y boosting son los del Lab 3 (si en tu Lab 3 ganó otra profundidad, cámbiala en `PROF_ARBOL`). La celda tarda de 1 a 3 min en Colab gratuito.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

PROF_ARBOL = 8          # la profundidad que ganó en validación en tu Lab 3
candidatos = {
    "logística (C = 1)": (LogisticRegression(C=1.0, max_iter=2000), True),
    f"árbol (max_depth = {PROF_ARBOL})": (DecisionTreeClassifier(max_depth=PROF_ARBOL, random_state=0), False),
    "bosque aleatorio": (RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=2, random_state=0), False),
    "boosting (HistGB)": (HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, early_stopping=False, random_state=0), False),
}
puntajes_va = {
    "mayoritario (siempre 0)": np.zeros(len(y_va)),
    "la semana se repite (|net_lag_168h|)": X_va["abs_net_lag_168h"].to_numpy(),
}
ajustados = {}
for nombre, (modelo, escala) in candidatos.items():
    ajustados[nombre] = modelo.fit(Z_tr if escala else X_tr, y_tr)
    puntajes_va[nombre] = modelo.predict_proba(Z_va if escala else X_va)[:, 1]
puntajes_va[f"SVC RBF (muestra 8 000, C = {mejor['C']:g}, γ = {mejor['γ']:g})"] = s_va_svc
puntajes_va["LinearSVC (hinge)"] = s_va_lin
puntajes_va["SGDClassifier (hinge)"] = s_va_sgd

comparativa = pd.DataFrame({"AP val": {k: average_precision_score(y_va, s) for k, s in puntajes_va.items()}})
comparativa = comparativa.sort_values("AP val", ascending=False)
comparativa.round(3)

In [ ]:
from sklearn.metrics import precision_score, recall_score


def umbral_presupuesto(p, presupuesto):
    # El umbral más bajo t (entre los valores de p) con mean(p >= t) <= presupuesto (igual que en el Lab 3)
    candidatos_t = np.unique(p)
    tasa = 1 - np.searchsorted(np.sort(p), candidatos_t, side="left") / len(p)
    return float(candidatos_t[np.argmax(tasa <= presupuesto)])


# El mejor en validación entre los que dan probabilidad (si ganara un SVM o un baseline, tomamos el siguiente)
ganador = next(k for k in comparativa.index if k in ajustados)
if ganador != comparativa.index[0]:
    print(f"Nota: el primero de la tabla es «{comparativa.index[0]}»; para el reporte usamos «{ganador}», que da probabilidades.")
PRESUPUESTO = 0.25
umbral_clf = umbral_presupuesto(puntajes_va[ganador], PRESUPUESTO)
print(f"ganador en val: {ganador} · umbral para alertar ≤ 25 %: {umbral_clf:.4f}")

# Test, una sola vez, con el modelo y el umbral congelados
p_te = ajustados[ganador].predict_proba(X_te)[:, 1]
alerta_te = p_te >= umbral_clf
print(f"test: AP = {average_precision_score(y_te, p_te):.3f} · alertas = {alerta_te.mean():.1%} · "
      f"precision = {precision_score(y_te, alerta_te):.3f} · recall = {recall_score(y_te, alerta_te):.3f}")

### L5d · ¿Una diferencia de 0.01 o 0.02 es real?
La AP de validación es **un** número calculado con **un** mes. Si el ganador le saca 0.02 al segundo, la pregunta honesta es si esa ventaja sobrevive a otro mes parecido. No tenemos otro julio, pero podemos remuestrear el que tenemos por **semanas** (*bootstrap* por bloques: las horas de una misma semana se parecen entre sí, así que remuestrear horas sueltas daría una falsa sensación de precisión).

In [ ]:
semana = ((va["hour"] - va["hour"].min()).dt.days // 7).to_numpy()      # bloques de 7 días desde el 1 de julio
bloques = [np.flatnonzero(semana == s) for s in np.unique(semana)]
segundo = comparativa.index[1]
logistica = "logística (C = 1)"

por_semana = pd.DataFrame({nombre: [average_precision_score(y_va[b], puntajes_va[nombre][b]) for b in bloques]
                           for nombre in [ganador, segundo, logistica]},
                          index=[f"semana {s + 1} ({len(b) // 40 // 24} días)" for s, b in enumerate(bloques)])
display(por_semana.round(3))

rng_b = np.random.default_rng(0)
difs = {segundo: [], logistica: []}
for _ in range(400):
    idx = np.concatenate([bloques[k] for k in rng_b.integers(0, len(bloques), len(bloques))])
    ap_g = average_precision_score(y_va[idx], puntajes_va[ganador][idx])
    for rival in difs:
        difs[rival].append(ap_g - average_precision_score(y_va[idx], puntajes_va[rival][idx]))
for rival, d in difs.items():
    d = np.array(d)
    print(f"{ganador} − {rival}: diferencia observada {comparativa.loc[ganador, 'AP val'] - comparativa.loc[rival, 'AP val']:+.3f} · "
          f"intervalo 90 % por semanas [{np.quantile(d, 0.05):+.3f}, {np.quantile(d, 0.95):+.3f}] · gana en {np.mean(d > 0):.0%} de las réplicas")

**Lee los resultados.** ¿La ventaja del ganador sobre el segundo lugar es más grande que el ruido entre semanas? ¿Y sobre la logística? ¿Qué más pesarías antes de desplegar?

**Tu respuesta:** _(escribe aquí)_

> ### ✅ Reacción en Zoom · fin de L5
> Chat waterfall: ¿desplegarías el ganador? Sí/no y una razón.
> Si te atoraste, reacciona con ✋ y escribe en el chat el número de la celda.

## L6 · Mini-reto (abierto): el mapa de sesgos
Para cada modelo de la tabla escribe en una línea su **sesgo inductivo** (qué supone del mundo) y si en MiBici esa apuesta le ayudó o le estorbó. Extra: ¿le ayuda al SVM RBF una muestra más grande (por ejemplo 15 000 filas)? Mide el tiempo antes de subirle.

No se califica: llévalo a la actividad de FigJam.

In [ ]:
# Tu exploración aquí

---
### Antes de entregar
1. *Reiniciar y ejecutar todo*. Las autocomprobaciones de L3 deben decir ✅.
2. Descarga el notebook (*Archivo*, *Descargar*, *Descargar .ipynb*) y súbelo a Moodle. No cambies el nombre de las funciones `graded`.